In [5]:
!pip install -q google-genai chromadb

In [8]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [15]:
import os, json, time, uuid
import chromadb
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
LLM_MODEL = "gemini-3.5-flash"       # change to a current Flash model if you get "model not found"
EMBED_MODEL = "gemini-embedding-001"

def llm(system, turns, json_mode=False):
    contents = [types.Content(role="model" if t["role"] == "assistant" else "user",
                              parts=[types.Part(text=t["content"])]) for t in turns]
    cfg = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json" if json_mode else "text/plain")
    return client.models.generate_content(model=LLM_MODEL, contents=contents, config=cfg).text

def embed(text, task):
    r = client.models.embed_content(model=EMBED_MODEL, contents=[text],
                                    config=types.EmbedContentConfig(task_type=task))
    return r.embeddings[0].values

In [16]:
class ShortTermMemory:
    def __init__(self, max_turns=6):
        self.turns, self.summary, self.max_turns = [], "", max_turns

    def add(self, role, content):
        self.turns.append({"role": role, "content": content})
        if len(self.turns) > self.max_turns:             # overflow -> summarize oldest 2 turns
            old, self.turns = self.turns[:2], self.turns[2:]
            text = "\n".join(f"{t['role']}: {t['content']}" for t in old)
            self.summary = llm(
                "Update the running conversation summary in max 3 sentences. "
                "Keep order IDs, amounts and the customer's problem.",
                [{"role": "user",
                  "content": f"Current summary: {self.summary or 'none'}\n\nNew turns:\n{text}"}])

In [17]:
class LongTermMemory:
    def __init__(self):
        self.col = chromadb.Client().get_or_create_collection(
            "episodes", metadata={"hnsw:space": "cosine"})
        self.profiles = {"U101": {"name": "Asha", "tier": "Gold", "language": "English"}}

    def save_episode(self, uid, text, status, importance, ts=None):
        self.col.add(ids=[str(uuid.uuid4())], documents=[text],
                     embeddings=[embed(text, "RETRIEVAL_DOCUMENT")],
                     metadatas=[{"user_id": uid, "status": status,
                                 "importance": float(importance), "ts": ts or time.time()}])

    def recall(self, uid, query, k=3):
        res = self.col.query(query_embeddings=[embed(query, "RETRIEVAL_QUERY")],
                             n_results=k * 3, where={"user_id": uid})
        scored = []
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
            sim = 1 - dist
            recency = 1 / (1 + (time.time() - meta["ts"]) / 86400)
            score = 0.6 * sim + 0.2 * recency + 0.2 * meta["importance"]
            scored.append((score, f"[{meta['status']}] {doc}"))
        return [d for _, d in sorted(scored, reverse=True)[:k]]

In [18]:
SYSTEM = """You are a customer support agent for a telecom company.
Use the customer profile and PAST CASES to personalise your reply and avoid asking
for details the customer already gave. If a past case is still open, acknowledge it
and say you are escalating. Never invent facts not present in the context.
Confirm with the customer before any refund. Be concise and polite."""

class SupportAgent:
    def __init__(self, ltm): self.ltm = ltm

    def start_session(self, uid):
        self.uid, self.stm = uid, ShortTermMemory()
        self.profile = self.ltm.profiles.get(uid, {})

    def handle(self, msg):
        past = self.ltm.recall(self.uid, msg)                 # LTM recall
        self.stm.add("user", msg)                             # STM update
        context = (f"CUSTOMER PROFILE: {self.profile}\n"
                   f"PAST CASES:\n" + ("\n".join(past) or "none") +
                   f"\nEARLIER IN THIS CHAT (summary): {self.stm.summary or 'none'}")
        reply = llm(SYSTEM + "\n\n" + context, self.stm.turns)
        self.stm.add("assistant", reply)
        return reply, past

    def end_session(self):                                    # consolidation into LTM
        transcript = "\n".join(f"{t['role']}: {t['content']}" for t in self.stm.turns)
        raw = llm('Return ONLY JSON: {"issue": str, "resolution": str, '
                  '"status": "open" or "resolved", "importance": float between 0 and 1}',
                  [{"role": "user", "content": f"Summary: {self.stm.summary}\n{transcript}"}],
                  json_mode=True)
        ep = json.loads(raw)
        self.ltm.save_episode(self.uid, f"{ep['issue']} -> {ep['resolution']}",
                              ep["status"], ep["importance"])
        return ep

In [19]:
ltm = LongTermMemory()
ltm.save_episode("U101", "Router keeps disconnecting every evening -> firmware updated",
                 "resolved", 0.4, ts=time.time() - 30 * 86400)
ltm.save_episode("U101", "Double charged for October bill -> refund requested, awaiting finance",
                 "open", 0.8, ts=time.time() - 12 * 86400)
print("Episodes in LTM:", ltm.col.count())

Episodes in LTM: 4


In [20]:
agent = SupportAgent(ltm)
agent.start_session("U101")

reply, recalled = agent.handle("Hi, I was charged twice again this month!")
print("RECALLED :", recalled)
print("AGENT    :", reply)

reply, _ = agent.handle("Yes please escalate it. How long will it take?")
print("\nAGENT    :", reply)

print("\nSAVED EPISODE:", agent.end_session())
print("Episodes in LTM:", ltm.col.count())

RECALLED : ['[open] Double charged for October bill -> refund requested, awaiting finance', '[open] Double charged for October bill -> refund requested, awaiting finance', '[resolved] Router keeps disconnecting every evening -> firmware updated']
AGENT    : Hello Asha,

Thank you for reaching out. I can see you currently have an open case regarding being double charged for your October bill, which is awaiting review by our finance team. 

I sincerely apologize for the continued trouble. I am escalating your open case right now to ensure our finance team prioritizes your request. Please note that we will confirm with you before any refund is processed.

Thank you for your patience!

AGENT    : Hi Asha,

I have escalated your case to our finance team for priority handling. While I don't have an exact timeframe available, our team will review it as quickly as possible. 

We will update you as soon as there is news and will confirm with you before any refund is issued. Thank you for your p

In [22]:
import time
from google.genai import errors

def with_retry(fn, tries=5):
    for i in range(tries):
        try:
            return fn()
        except errors.ServerError as e:            # 500/503: server overloaded
            wait = 2 ** i                          # 1, 2, 4, 8, 16 seconds
            print(f"Server busy ({e.code}), retrying in {wait}s...")
            time.sleep(wait)
    return fn()                                    # last attempt raises the real error

def llm(system, turns, json_mode=False):
    contents = [types.Content(role="model" if t["role"] == "assistant" else "user",
                              parts=[types.Part(text=t["content"])]) for t in turns]
    cfg = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json" if json_mode else "text/plain")
    return with_retry(lambda: client.models.generate_content(
        model=LLM_MODEL, contents=contents, config=cfg).text)

def embed(text, task):
    r = with_retry(lambda: client.models.embed_content(
        model=EMBED_MODEL, contents=[text],
        config=types.EmbedContentConfig(task_type=task)))
    return r.embeddings[0].values

In [23]:
agent.start_session("U101")
reply, recalled = agent.handle("Any update on my billing issue?")
print("RECALLED :", recalled)
print("AGENT    :", reply)

RECALLED : ['[open] Customer reported being double charged for their monthly bill. -> Escalated the open case to the finance team for priority handling and informed the customer that they will be updated once reviewed.', '[open] Double charged for October bill -> refund requested, awaiting finance', '[open] Double charged for October bill -> refund requested, awaiting finance']
AGENT    : Hello Asha,

Thank you for following up. I can see your open case regarding the double charge on your October bill, which is currently awaiting review by our finance team for your refund request. 

I am escalating this case now to help speed up the process for you. We will reach out with an update as soon as the finance team reviews it. 

Thank you for your patience!


In [24]:
!pip install -q gradio

In [ ]:
import gradio as gr

ltm.profiles["U102"] = {"name": "Ravi", "tier": "Silver", "language": "English"}  # new customer, no history

agent = SupportAgent(ltm)
session = {"uid": None}          # simple global state (fine for a single-user lab demo)

def ltm_table(uid):
    res = ltm.col.get(where={"user_id": uid})
    rows = sorted(zip(res["documents"], res["metadatas"]), key=lambda x: -x[1]["ts"])
    return [[m["status"], round(m["importance"], 2), d] for d, m in rows]

def stm_view():
    if session["uid"] is None:
        return "**Short-term memory:** empty"
    turns = "\n".join(f"- *{t['role']}*: {t['content'][:80]}" for t in agent.stm.turns)
    return f"**Short-term memory**\n\nSummary: {agent.stm.summary or '–'}\n\nLast turns:\n{turns or '–'}"

def chat(msg, history, uid):
    history = history or []
    if not msg.strip():
        return history, "", "**Recalled from LTM:** –", stm_view()
    if session["uid"] != uid:                       # start a new session if needed
        agent.start_session(uid)
        session["uid"] = uid
    try:
        reply, recalled = agent.handle(msg)
    except Exception as e:
        reply, recalled = f"⚠️ Error: {e}", []
    history = history + [{"role": "user", "content": msg},
                         {"role": "assistant", "content": reply}]
    rec = "\n".join(f"- {r}" for r in recalled) or "none"
    return history, "", f"**Recalled from LTM:**\n\n{rec}", stm_view()

def end_session(uid):
    if session["uid"] is None or not agent.stm.turns:
        return [], "Nothing to save yet.", ltm_table(uid), stm_view()
    try:
        ep = agent.end_session()
        msg = f"✅ Saved to LTM: **{ep['issue']}** → {ep['resolution']} ({ep['status']}, importance {ep['importance']})"
    except Exception as e:
        msg = f"⚠️ Could not save: {e}"
    session["uid"] = None                           # next message starts a fresh session
    return [], msg, ltm_table(uid), stm_view()

def switch_user(uid):
    session["uid"] = None
    return [], "**Recalled from LTM:** –", ltm_table(uid), stm_view(), ""

try:
    chatbot = gr.Chatbot(type="messages", height=380, label="Support chat")
except TypeError:                                   # newer Gradio versions
    chatbot = gr.Chatbot(height=380, label="Support chat")

with gr.Blocks(title="Support Agent with Memory") as demo:
    gr.Markdown("# 🎧 Customer Support Agent: Short-term + Long-term Memory")
    with gr.Row():
        with gr.Column(scale=3):
            uid = gr.Dropdown(["U101", "U102"], value="U101", label="Customer ID")
            chatbot.render()
            box = gr.Textbox(placeholder="Type your complaint and press Enter...", show_label=False)
            with gr.Row():
                send = gr.Button("Send", variant="primary")
                end = gr.Button("End session & save to LTM")
        with gr.Column(scale=2):
            recalled_md = gr.Markdown("**Recalled from LTM:** –")
            stm_md = gr.Markdown(stm_view())
            status = gr.Markdown()
            table = gr.Dataframe(headers=["status", "importance", "episode"],
                                 value=ltm_table("U101"), interactive=False,
                                 wrap=True, label="Long-term memory (this customer)")

    box.submit(chat, [box, chatbot, uid], [chatbot, box, recalled_md, stm_md])
    send.click(chat, [box, chatbot, uid], [chatbot, box, recalled_md, stm_md])
    end.click(end_session, [uid], [chatbot, status, table, stm_md])
    uid.change(switch_user, [uid], [chatbot, recalled_md, table, stm_md, status])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6fa5ea454093560869.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
